In [ ]:
# =============================================================================
# SECTION 1: ENVIRONMENT SETUP -- LOAD NOTEBOOKS 70/71'S REAL OUTPUTS
# =============================================================================
import json
import os
from pathlib import Path
from datetime import datetime, timezone


def _section(title: str) -> None:
    bar = "=" * 78
    print(f"\n{bar}\n{title}\n{bar}")


_section("SECTION 1: Environment Setup -- Load Notebooks 70/71's Real Outputs")

PROJECT_ROOT = Path(r"C:\Users\rnand\Downloads\amex-default-prediction\AMEX_Enterprise_Credit_Risk_Platform")
P14_ROOT = PROJECT_ROOT / "Phase5_Customer_Business_Intelligence" / "Problem14_Executive_Decision_Support_Dashboard"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

CONFIG_PATH = ARTIFACTS_DIR / "project_config.json"
NB70_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_70_summary.json"
NB71_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_71_summary.json"
for _p, _fix in [
    (CONFIG_PATH, "run 01_business_understanding.ipynb first."),
    (NB70_SUMMARY_PATH, "run 70_executive_dashboard_business_understanding.ipynb first."),
    (NB71_SUMMARY_PATH, "run 71_executive_dashboard_modeling.ipynb first."),
]:
    if not _p.exists():
        raise FileNotFoundError(f"{_p.name} not found in its expected location.\nFix: {_fix}")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PROJECT_CONFIG = json.load(f)
with open(NB70_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB70_SUMMARY = json.load(f)
with open(NB71_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB71_SUMMARY = json.load(f)

WARP_THREAD_COUNT = PROJECT_CONFIG["resource_limits"]["warp_thread_count"]
MAX_RAM_BYTES = PROJECT_CONFIG["resource_limits"]["max_ram_bytes"]
RANDOM_SEED = NB71_SUMMARY["random_seed"]
PILLAR_DIRS = {k: Path(v) for k, v in PROJECT_CONFIG["pillar_dirs"].items()}

POLICY_PATH = Path(NB70_SUMMARY["policy_path"])
with open(POLICY_PATH, "r", encoding="utf-8") as f:
    EXECUTIVE_DASHBOARD_POLICY = json.load(f)
PRIOR_PROBLEMS_REGISTRY = {int(k): v for k, v in EXECUTIVE_DASHBOARD_POLICY["prior_problems_registry"].items()}
EXPECTED_FOUNDATIONAL_PROBLEMS = set(EXECUTIVE_DASHBOARD_POLICY["expected_foundational_problems"])
EXPECTED_RESERVE_OPTIMIZATION_PROBLEMS = set(EXECUTIVE_DASHBOARD_POLICY["expected_reserve_optimization_problems"])

DASHBOARD_DATA_PATH = Path(NB71_SUMMARY["dashboard_data_path"])
if not DASHBOARD_DATA_PATH.exists():
    raise FileNotFoundError(f"{DASHBOARD_DATA_PATH} not found.\nFix: re-run Notebook 71.")
with open(DASHBOARD_DATA_PATH, "r", encoding="utf-8") as f:
    PERSISTED_DASHBOARD_DATA = json.load(f)

print(f"Real persisted TOTAL_PLATFORM_NET_VALUE_USD (Notebook 71): "
      f"${PERSISTED_DASHBOARD_DATA['total_platform_net_value_usd']:,.2f}")
print("\n✅ Section 1 complete.")


# =============================================================================
# SECTION 2: LIBRARY IMPORTS
# =============================================================================
_section("SECTION 2: Library Imports")

missing = []
try:
    import polars as pl
except ImportError:
    missing.append("polars")
try:
    import importlib.util
except ImportError:
    missing.append("importlib")
try:
    from fastapi.testclient import TestClient
except ImportError:
    missing.append("fastapi[testclient]")
if missing:
    raise ImportError(f"Missing required libraries: {missing}. Install with: pip install {' '.join(missing)}")

print("✅ Section 2 complete.")


# =============================================================================
# SECTION 3: INDEPENDENT REPRODUCTION -- REBUILD THE ENTIRE AGGREGATION FROM
#            SCRATCH, FRESH KERNEL, RE-READING ALL 13 REAL SUMMARY JSONS
# =============================================================================
_section("SECTION 3: Independent Reproduction -- Rebuild the Aggregation From Scratch")


def _extract(problem_summaries: dict, path):
    if path is None:
        return None
    source_key, *nested = path
    node = problem_summaries.get(source_key)
    for key in nested:
        if not isinstance(node, dict):
            return None
        node = node.get(key)
    return node


_repro_loaded, _repro_missing = {}, []
for _pnum, _entry in PRIOR_PROBLEMS_REGISTRY.items():
    _repro_loaded[_pnum] = {}
    for _key, _fname in _entry["summary_jsons"].items():
        _path = ARTIFACTS_DIR / _fname
        if not _path.exists():
            _repro_missing.append(f"Problem {_pnum}: {_fname}")
            continue
        with open(_path, "r", encoding="utf-8") as f:
            _repro_loaded[_pnum][_key] = json.load(f)
if _repro_missing:
    raise FileNotFoundError("Missing real summary JSON(s):\n  " + "\n  ".join(_repro_missing))

REPRO_ROWS = []
for _pnum in sorted(PRIOR_PROBLEMS_REGISTRY):
    _entry = PRIOR_PROBLEMS_REGISTRY[_pnum]
    _summaries = _repro_loaded[_pnum]
    _financial_val = _extract(_summaries, _entry["financial_field"])
    _status_val = _extract(_summaries, _entry["status_field"])
    REPRO_ROWS.append({
        "problem_number": _pnum, "category": _entry["category"],
        "financial_value_usd": (round(_financial_val, 2) if isinstance(_financial_val, (int, float))
                                 else None),
        "recommended_for_production": bool(_status_val) if isinstance(_status_val, bool) else None,
    })

REPRO_INCLUDED = set(
    r["problem_number"] for r in REPRO_ROWS if r["category"] == "value_creation" and
    r["recommended_for_production"] is True
)
REPRO_TOTAL_PLATFORM_NET_VALUE_USD = round(
    sum(r["financial_value_usd"] for r in REPRO_ROWS if r["problem_number"] in REPRO_INCLUDED), 2
)

_persisted_total = PERSISTED_DASHBOARD_DATA["total_platform_net_value_usd"]
_total_diff = abs(REPRO_TOTAL_PLATFORM_NET_VALUE_USD - _persisted_total)
REPRODUCTION_PASSED = bool(_total_diff < 0.01 and sorted(REPRO_INCLUDED) == PERSISTED_DASHBOARD_DATA[
    "included_problems"])

print(f"Reproduced TOTAL_PLATFORM_NET_VALUE_USD : ${REPRO_TOTAL_PLATFORM_NET_VALUE_USD:,.2f}")
print(f"Persisted  TOTAL_PLATFORM_NET_VALUE_USD : ${_persisted_total:,.2f}")
print(f"Reproduced included-problem set : {sorted(REPRO_INCLUDED)}")
print(f"Persisted  included-problem set : {PERSISTED_DASHBOARD_DATA['included_problems']}")
print(f"reproduction_passed: {'PASS' if REPRODUCTION_PASSED else 'FAIL'}")
if not REPRODUCTION_PASSED:
    raise RuntimeError("Independent reproduction did not match Notebook 71's persisted total -- see above.")
print("\n✅ Section 3 complete.")


# =============================================================================
# SECTION 4: CROSS-CHECK THE PERSISTED TABLE ROW-BY-ROW AGAINST THE FRESH
#            REPRODUCTION
# =============================================================================
_section("SECTION 4: Cross-Check the Persisted Table Row-by-Row")

TABLE_PATH = Path(NB71_SUMMARY["table_path"])
if not TABLE_PATH.exists():
    raise FileNotFoundError(f"{TABLE_PATH} not found.\nFix: re-run Notebook 71.")
PERSISTED_TABLE = pl.read_parquet(TABLE_PATH)

_row_mismatches = []
for _repro_row in REPRO_ROWS:
    _persisted_row = PERSISTED_TABLE.filter(pl.col("problem_number") == _repro_row["problem_number"])
    if _persisted_row.height != 1:
        _row_mismatches.append(f"Problem {_repro_row['problem_number']}: not found in persisted table")
        continue
    _p = _persisted_row.row(0, named=True)
    _fin_match = (
        (_p["financial_value_usd"] is None and _repro_row["financial_value_usd"] is None) or
        (_p["financial_value_usd"] is not None and _repro_row["financial_value_usd"] is not None and
         abs(_p["financial_value_usd"] - _repro_row["financial_value_usd"]) < 0.01)
    )
    if not _fin_match or _p["category"] != _repro_row["category"]:
        _row_mismatches.append(f"Problem {_repro_row['problem_number']}: mismatch (persisted="
                                f"{_p['financial_value_usd']}, reproduced={_repro_row['financial_value_usd']})")

PROFILE_VERIFIED = bool(len(_row_mismatches) == 0)
print(f"Rows cross-checked: {len(REPRO_ROWS)} / mismatches: {len(_row_mismatches)}")
for _m in _row_mismatches:
    print(f"  ❌ {_m}")
print(f"profile_verified: {'PASS' if PROFILE_VERIFIED else 'FAIL'}")
if not PROFILE_VERIFIED:
    raise RuntimeError("Persisted table does not match the fresh reproduction -- see mismatches above.")
print("\n✅ Section 4 complete.")


# =============================================================================
# SECTION 5: RE-VALIDATE BOTH HARD-GATING KPIS, FRESH
# =============================================================================
_section("SECTION 5: Re-Validate Both Hard-Gating KPIs, Fresh")

REPRO_EXCLUDED = set(range(1, 14)) - REPRO_INCLUDED
_check_a = (REPRO_INCLUDED | REPRO_EXCLUDED) == set(range(1, 14)) and not (REPRO_INCLUDED & REPRO_EXCLUDED)
_check_b = (EXPECTED_FOUNDATIONAL_PROBLEMS | EXPECTED_RESERVE_OPTIMIZATION_PROBLEMS).issubset(REPRO_EXCLUDED)
_value_creation_excluded = REPRO_EXCLUDED - EXPECTED_FOUNDATIONAL_PROBLEMS - EXPECTED_RESERVE_OPTIMIZATION_PROBLEMS
_check_c = all(
    next(r for r in REPRO_ROWS if r["problem_number"] == p)["recommended_for_production"] is not True
    for p in _value_creation_excluded
)
AGGREGATION_SCOPE_CORRECTNESS_PASSED = bool(_check_a and _check_b and _check_c)
AGGREGATION_COMPLETENESS_PASSED = bool(
    len(REPRO_ROWS) == 13 and all(
        r["financial_value_usd"] is not None for r in REPRO_ROWS if r["category"] == "value_creation"
    )
)

print(f"aggregation_completeness (fresh)      : {'PASS' if AGGREGATION_COMPLETENESS_PASSED else 'FAIL'}")
print(f"aggregation_scope_correctness (fresh) : {'PASS' if AGGREGATION_SCOPE_CORRECTNESS_PASSED else 'FAIL'}")
ALL_HARD_GATES_PASSED = bool(AGGREGATION_COMPLETENESS_PASSED and AGGREGATION_SCOPE_CORRECTNESS_PASSED)
if not ALL_HARD_GATES_PASSED:
    raise RuntimeError("One or more hard-gating KPIs failed on fresh reproduction -- see above.")
print("\n✅ Section 5 complete.")


# =============================================================================
# SECTION 6: DEPLOYMENT SCOPE NOTE
# =============================================================================
_section("SECTION 6: Deployment Scope Note")

RECOMMENDED_FOR_PRODUCTION = ALL_HARD_GATES_PASSED
print(
    "Problem 14's deliverable is a precomputed BI aggregation artifact (13 real problem summaries rolled "
    "up into one executive JSON), not a per-request live model -- consistent with Problems 12 and 13's "
    "own precomputed-lookup architecture decision. The deployed service below serves this real, "
    "already-computed rollup; it does not re-run any of the 13 problems' own pipelines."
)
print(f"recommended_for_production: {RECOMMENDED_FOR_PRODUCTION}")
print("\n✅ Section 6 complete.")


# =============================================================================
# SECTION 7: PERSIST DEPLOYMENT POLICY
# =============================================================================
_section("SECTION 7: Persist Deployment Policy")

if "executive_dashboard_docs" in PILLAR_DIRS:
    P14_DOCS_DIR = PILLAR_DIRS["executive_dashboard_docs"]
else:
    P14_DOCS_DIR = P14_ROOT / "docs"
    print(f"NOTE: 'executive_dashboard_docs' not in pillar_dirs -- using fallback: {P14_DOCS_DIR}")
P14_DOCS_DIR.mkdir(parents=True, exist_ok=True)

EXECUTIVE_DEPLOYMENT_POLICY = {
    "problem_number": 14, "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "dashboard_data_path": str(DASHBOARD_DATA_PATH),
    "total_platform_net_value_usd": REPRO_TOTAL_PLATFORM_NET_VALUE_USD,
    "included_problems": sorted(REPRO_INCLUDED), "excluded_problems": sorted(REPRO_EXCLUDED),
    "recommended_for_production": RECOMMENDED_FOR_PRODUCTION,
    "reproduction_passed": REPRODUCTION_PASSED, "profile_verified": PROFILE_VERIFIED,
    "random_seed": RANDOM_SEED,
}
deployment_policy_path = P14_DOCS_DIR / "executive_dashboard_deployment_policy.json"
with open(deployment_policy_path, "w", encoding="utf-8") as f:
    json.dump(EXECUTIVE_DEPLOYMENT_POLICY, f, indent=2)
print(f"Deployment policy written to: {deployment_policy_path}")
print("\n✅ Section 7 complete.")


# =============================================================================
# SECTION 8: GENERATE executive_dashboard_service.py -- RUNNABLE FASTAPI
#            LOOKUP SERVICE WITH AUTH
# =============================================================================
_section("SECTION 8: Generate executive_dashboard_service.py")

if "executive_dashboard_src" in PILLAR_DIRS:
    API_SUBDIR = PILLAR_DIRS["executive_dashboard_src"]
else:
    API_SUBDIR = P14_ROOT / "src"
API_SUBDIR.mkdir(parents=True, exist_ok=True)

_policy_path_str = str(deployment_policy_path)
_data_path_str = str(DASHBOARD_DATA_PATH)

EXECUTIVE_SERVICE_TEMPLATE = "\n".join([
    "# AMEX Enterprise Credit Risk Platform -- Executive Decision Support Dashboard API.",
    "# Auto-generated by 72_executive_dashboard_validation_deployment.ipynb.",
    "# Serves the real, precomputed roll-up of all 13 prior problems' real results (Notebook 71's real",
    "# BI aggregation layer). Every endpoint except /health requires a valid X-API-Key header.",
    "# Run with:",
    "#     uvicorn executive_dashboard_service:app --host 0.0.0.0 --port 8014",
    "import json",
    "import logging",
    "import os",
    "import secrets",
    "from pathlib import Path",
    "",
    "from fastapi import Depends, FastAPI, HTTPException, Security",
    "from fastapi.security import APIKeyHeader",
    "",
    "_auth_logger = logging.getLogger(__name__ + \".auth\")",
    "_DEV_DEFAULT_API_KEY = \"dev-only-CHANGE-ME-before-deploying\"",
    "_api_key_header = APIKeyHeader(name=\"X-API-Key\", auto_error=False)",
    "",
    "",
    "def _configured_api_key() -> str:",
    "    key = os.environ.get(\"API_KEY\")",
    "    if not key:",
    "        _auth_logger.warning(",
    "            \"API_KEY is not set -- falling back to the published dev-only default. Set API_KEY \"",
    "            \"before deploying this service anywhere reachable by anyone but you.\"",
    "        )",
    "        return _DEV_DEFAULT_API_KEY",
    "    return key",
    "",
    "",
    "def require_api_key(presented: str = Security(_api_key_header)) -> str:",
    "    expected = _configured_api_key()",
    "    if not presented or not secrets.compare_digest(presented, expected):",
    "        raise HTTPException(status_code=401, detail=\"Missing or invalid X-API-Key header.\")",
    "    return presented",
    "",
    "",
    "POLICY_PATH = Path(os.environ.get(\"AMEX_P14_POLICY_PATH\", r\"__POLICY_PATH_TOKEN__\"))",
    "DASHBOARD_DATA_PATH = Path(os.environ.get(\"AMEX_P14_DATA_PATH\", r\"__DATA_PATH_TOKEN__\"))",
    "with open(POLICY_PATH, \"r\", encoding=\"utf-8\") as _f:",
    "    _POLICY = json.load(_f)",
    "with open(DASHBOARD_DATA_PATH, \"r\", encoding=\"utf-8\") as _f:",
    "    _DASHBOARD_DATA = json.load(_f)",
    "",
    "_ROWS_BY_PROBLEM = {row[\"problem_number\"]: row for row in _DASHBOARD_DATA[\"rows\"]}",
    "",
    "app = FastAPI(",
    "    title=\"AMEX Enterprise Credit Risk Platform -- Executive Decision Support Dashboard API\",",
    "    description=\"Serves the real, precomputed roll-up of all 13 prior problems. Every endpoint except \"",
    "                \"/health requires a valid X-API-Key header.\",",
    "    version=\"1.0.0\",",
    ")",
    "",
    "",
    "@app.get(\"/health\")",
    "def health():",
    "    return {\"status\": \"ok\", \"problems_loaded\": len(_ROWS_BY_PROBLEM)}",
    "",
    "",
    "@app.get(\"/executive-summary\", dependencies=[Depends(require_api_key)])",
    "def executive_summary():",
    "    return _DASHBOARD_DATA",
    "",
    "",
    "@app.get(\"/problem/{problem_number}\", dependencies=[Depends(require_api_key)])",
    "def get_problem(problem_number: int):",
    "    row = _ROWS_BY_PROBLEM.get(problem_number)",
    "    if row is None:",
    "        raise HTTPException(status_code=404, detail=f\"No such problem_number={problem_number!r}. \"",
    "                                                     f\"Valid range: 1-13.\")",
    "    return row",
    "",
])
EXECUTIVE_SERVICE_SOURCE = (
    EXECUTIVE_SERVICE_TEMPLATE
    .replace("__POLICY_PATH_TOKEN__", _policy_path_str)
    .replace("__DATA_PATH_TOKEN__", _data_path_str)
)

service_py_path = API_SUBDIR / "executive_dashboard_service.py"
with open(service_py_path, "w", encoding="utf-8") as f:
    f.write(EXECUTIVE_SERVICE_SOURCE)
compile(EXECUTIVE_SERVICE_SOURCE, str(service_py_path), "exec")
print(f"Generated {len(EXECUTIVE_SERVICE_SOURCE.splitlines())} lines, syntax-checked OK.")
print(f"Saved -> {service_py_path}")
print("\n✅ Section 8 complete.")


# =============================================================================
# SECTION 9: LIVE SELF-TEST -- IMPORT THE GENERATED SERVICE & DRIVE IT
# =============================================================================
_section("SECTION 9: Live Self-Test -- Import the Generated Service & Drive It")

os.environ["AMEX_P14_POLICY_PATH"] = str(deployment_policy_path)
os.environ["AMEX_P14_DATA_PATH"] = str(DASHBOARD_DATA_PATH)
_TEST_API_KEY = "pytest-only-test-key"
os.environ["API_KEY"] = _TEST_API_KEY
_spec = importlib.util.spec_from_file_location("amex_executive_service", str(service_py_path))
_service_module = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_service_module)
client = TestClient(_service_module.app)
_auth_headers = {"X-API-Key": _TEST_API_KEY}

_health_resp = client.get("/health")
assert _health_resp.status_code == 200
print(f"GET /health                (no key)   -> {_health_resp.status_code}  {_health_resp.json()}")

_unauth_resp = client.get("/executive-summary")
assert _unauth_resp.status_code == 401
print(f"GET /executive-summary     (no key, should reject) -> {_unauth_resp.status_code}")

_summary_resp = client.get("/executive-summary", headers=_auth_headers)
assert _summary_resp.status_code == 200
_live_total = _summary_resp.json()["total_platform_net_value_usd"]
_live_total_match = abs(_live_total - REPRO_TOTAL_PLATFORM_NET_VALUE_USD) < 0.01
print(f"GET /executive-summary     (with key) -> {_summary_resp.status_code}  total=${_live_total:,.2f} "
      f"({'PASS' if _live_total_match else 'FAIL'} vs. reproduced total)")

API_SELF_TEST_ROWS_PASSED = [_live_total_match]
for _pnum in range(1, 14):
    _resp = client.get(f"/problem/{_pnum}", headers=_auth_headers)
    assert _resp.status_code == 200, f"/problem/{_pnum} returned {_resp.status_code}: {_resp.text}"
    _live_row = _resp.json()
    _expected_row = next(r for r in REPRO_ROWS if r["problem_number"] == _pnum)
    _row_ok = (
        _live_row["category"] == _expected_row["category"] and
        ((_live_row["financial_value_usd"] is None and _expected_row["financial_value_usd"] is None) or
         (_live_row["financial_value_usd"] is not None and _expected_row["financial_value_usd"] is not None and
          abs(_live_row["financial_value_usd"] - _expected_row["financial_value_usd"]) < 0.01))
    )
    API_SELF_TEST_ROWS_PASSED.append(_row_ok)
    print(f"GET /problem/{_pnum:<2}              (with key) -> {_resp.status_code}  "
          f"category={_live_row['category']:<20} {'PASS' if _row_ok else 'FAIL'}")

_missing_resp = client.get("/problem/99", headers=_auth_headers)
assert _missing_resp.status_code == 404, "/problem/99 should 404"
print(f"GET /problem/99            (should 404) -> {_missing_resp.status_code}")

_unauth_problem_resp = client.get("/problem/1")
assert _unauth_problem_resp.status_code == 401, "/problem/1 without a key should be rejected"
print(f"GET /problem/1             (no key, should reject) -> {_unauth_problem_resp.status_code}")

API_SELF_TEST_PASSED = bool(all(API_SELF_TEST_ROWS_PASSED)) and len(API_SELF_TEST_ROWS_PASSED) == 14
if not API_SELF_TEST_PASSED:
    raise RuntimeError("Notebook 72's API self-test FAILED -- see checks above. Not safe to proceed.")
print("\n✅ Section 9 complete -- auth rejects unkeyed calls, all 13 real problem rows match the fresh "
      "reproduction, an unknown problem_number correctly 404s.")


# =============================================================================
# SECTION 10: GENERATE .env.example & requirements-api.txt
# =============================================================================
_section("SECTION 10: Generate .env.example & requirements-api.txt")

_env_example = "\n".join([
    "# Copy to .env and fill in real values before deploying.",
    "API_KEY=dev-only-CHANGE-ME-before-deploying",
    f"AMEX_P14_POLICY_PATH={deployment_policy_path}",
    f"AMEX_P14_DATA_PATH={DASHBOARD_DATA_PATH}",
    "",
])
(API_SUBDIR / ".env.example").write_text(_env_example, encoding="utf-8")
_requirements_api = "\n".join(["fastapi>=0.110", "uvicorn>=0.29", "pydantic>=2.0", ""])
(API_SUBDIR / "requirements-api.txt").write_text(_requirements_api, encoding="utf-8")
print(f"Wrote: {API_SUBDIR / '.env.example'}")
print(f"Wrote: {API_SUBDIR / 'requirements-api.txt'}")
print("\n✅ Section 10 complete.")


# =============================================================================
# SECTION 11: VERIFICATION -- INTEGRITY CHECKS
# =============================================================================
_section("SECTION 11: Verification -- Integrity Checks")


def _check(label, condition, detail=""):
    status = "PASS" if condition else "FAIL"
    print(f"  [{status}] {label}" + (f" -- {detail}" if detail and not condition else ""))
    return condition


_all_checks_passed = True
_all_checks_passed &= _check("Reproduction matches Notebook 71's persisted total", REPRODUCTION_PASSED)
_all_checks_passed &= _check("Persisted table matches fresh reproduction row-by-row", PROFILE_VERIFIED)
_all_checks_passed &= _check("aggregation_completeness (fresh)", AGGREGATION_COMPLETENESS_PASSED)
_all_checks_passed &= _check("aggregation_scope_correctness (fresh)", AGGREGATION_SCOPE_CORRECTNESS_PASSED)
_all_checks_passed &= _check("Generated service compiles", True)
_all_checks_passed &= _check("API self-test passed (all 13 rows + auth + 404)", API_SELF_TEST_PASSED)

if not _all_checks_passed:
    raise RuntimeError("One or more Notebook 72 verification checks failed. See ❌ line above.")
print("\nAll Notebook 72 checks passed.")
print("\n✅ Section 11 complete.")


# =============================================================================
# SECTION 12: WRITE NOTEBOOK 72 SUMMARY
# =============================================================================
_section("SECTION 12: Write Notebook 72 Summary")

notebook_72_summary = {
    "notebook": "72_executive_dashboard_validation_deployment",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "deployment_policy_path": str(deployment_policy_path), "service_py_path": str(service_py_path),
    "reproduction_passed": REPRODUCTION_PASSED, "profile_verified": PROFILE_VERIFIED,
    "reproduced_total_platform_net_value_usd": REPRO_TOTAL_PLATFORM_NET_VALUE_USD,
    "all_hard_gates_passed": ALL_HARD_GATES_PASSED, "recommended_for_production": RECOMMENDED_FOR_PRODUCTION,
    "api_self_test_passed": API_SELF_TEST_PASSED, "random_seed": RANDOM_SEED,
}
nb72_summary_path = ARTIFACTS_DIR / "notebook_72_summary.json"
with open(nb72_summary_path, "w", encoding="utf-8") as f:
    json.dump(notebook_72_summary, f, indent=2)
print(f"Summary written to: {nb72_summary_path}")
print("\n✅ Section 12 complete.")

print(
    "\n🎯 Notebook 72 (Validation & Deployment) complete. Independent reproduction matches, both hard-"
    "gating KPIs re-verified fresh, real auth-protected executive_dashboard_service.py self-tested "
    "against all 13 real problem rows. Next: Notebook 73 (Financial Impact, Reporting & Packaging -- "
    "the grand finale)."
)
